#### Imports / setup

In [ ]:
import os
from pathlib import Path

import joblib
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import xgboost as xgb
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from torch.utils.data import DataLoader, TensorDataset

os.chdir("..")

FEATURES_PATH = Path("data/features/features.parquet")
MODELS_DIR = Path("src/models")
SEED = 42
TEST_SIZE = 0.2
VAL_SIZE = 0.1

print("Ready.")


#### Load features

In [ ]:
df = pd.read_parquet(FEATURES_PATH)

X = df.drop(columns=["default"])
y = df["default"]

print(f"Features: {X.shape[1]}  |  Rows: {len(X):,}")
print(f"Default rate: {y.mean():.1%}")


#### Train/val/test splits

In [ ]:
X_temp, X_test, y_temp, y_test = train_test_split(
    X, y, test_size=TEST_SIZE,
    random_state=SEED, stratify=y
)
X_train, X_val, y_train, y_val = train_test_split(
    X_temp, y_temp, test_size=VAL_SIZE,
    random_state=SEED, stratify=y_temp
)

print(f"Train: {len(X_train):,}  |  Val: {len(X_val):,}  |  Test: {len(X_test):,}")


#### Scale features

In [ ]:
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_val_scaled   = scaler.transform(X_val)
X_test_scaled  = scaler.transform(X_test)
print("Scaling done.")

## Models:
 #### Standard Log Regression

In [ ]:
lr = LogisticRegression(max_iter=1000, random_state=SEED, class_weight="balanced")
lr.fit(X_train_scaled, y_train)

lr_val_auc = roc_auc_score(y_val, lr.predict_proba(X_val_scaled)[:, 1])
print(f"Logistic Regression — Val AUC: {lr_val_auc:.4f}")

#### XGBoost

In [ ]:
xgb_model = xgb.XGBClassifier(
    n_estimators=300,
    max_depth=5,
    learning_rate=0.05,
    subsample=0.8,
    colsample_bytree=0.8,
    # scale_pos_weight=(y_train == 0).sum() / (y_train == 1).sum(),
    random_state=SEED,
    eval_metric="auc",
    early_stopping_rounds=20,
    verbosity=0
)
xgb_model.fit(
    X_train, y_train,
    eval_set=[(X_val, y_val)],
    verbose=False
)

xgb_val_auc = roc_auc_score(y_val, xgb_model.predict_proba(X_val)[:, 1])
print(f"XGBoost — Val AUC: {xgb_val_auc:.4f}")

### NN

In [ ]:
class CreditNN(nn.Module):
    def __init__(self, input_dim):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(input_dim, 128),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(128, 64),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(64, 1),
        )

    def forward(self, x):
        return self.net(x).squeeze(-1)

# Prepare tensors
X_tr = torch.tensor(X_train_scaled, dtype=torch.float32)
y_tr = torch.tensor(y_train.values, dtype=torch.float32)
X_v  = torch.tensor(X_val_scaled, dtype=torch.float32)

train_loader = DataLoader(TensorDataset(X_tr, y_tr), batch_size=512, shuffle=True)

model = CreditNN(X_train_scaled.shape[1])
optimizer = torch.optim.Adam(model.parameters(), lr=0.001)
pos_weight = torch.tensor([(y_train == 0).sum() / (y_train == 1).sum()], dtype=torch.float32)
criterion = nn.BCEWithLogitsLoss(pos_weight=pos_weight)

# Training loop
for epoch in range(10):
    model.train()
    for xb, yb in train_loader:
        optimizer.zero_grad()
        logits = model(xb)
        loss = criterion(logits, yb)
        loss.backward()
        optimizer.step()

model.eval()
with torch.no_grad():
    nn_probs = torch.sigmoid(model(X_v)).numpy()

nn_val_auc = roc_auc_score(y_val, nn_probs)
print(f"Neural Network — Val AUC: {nn_val_auc:.4f}")


#### Comparison

In [ ]:
results = pd.DataFrame({
    "Model": ["Logistic Regression", "XGBoost", "Neural Network"],
    "Val AUC": [lr_val_auc, xgb_val_auc, nn_val_auc]
}).sort_values("Val AUC", ascending=False)

print(results.to_string(index=False))


### Save Models

In [ ]:
MODELS_DIR.mkdir(parents=True, exist_ok=True)

# Save all three models plus held-out test artifacts for notebook 4.
joblib.dump(lr, MODELS_DIR / "logistic_regression.joblib")
joblib.dump(xgb_model, MODELS_DIR / "xgboost_model.joblib")
joblib.dump(scaler, MODELS_DIR / "scaler.joblib")
torch.save(model.state_dict(), MODELS_DIR / "neural_net.pt")

X_test.to_parquet(MODELS_DIR / "X_test.parquet", index=False)
y_test.to_frame(name="default").to_parquet(MODELS_DIR / "y_test.parquet", index=False)

print("All models saved.")
print(f"Held-out test set saved: {len(X_test):,} rows")
